In [ ]:
import cv2
import numpy as np
from pathlib import Path
import random

def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []

    for dir in dirs:
        label = dir.name  # Label dari nama folder
        for file in dir.glob('*.jpg'):
            try:
                img = cv2.imread(str(file))
                if img is not None:
                    img_list.append((img, label))
            except OSError:
                print(f"Skipped corrupted image: {file}")
    return img_list

# Path dataset
train_dir = "FaceShape/training_set"
test_dir = "FaceShape/testing_set"

# Load dataset
train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)


In [ ]:
def detectFace(image):
    """Mendeteksi wajah menggunakan Haar Cascade dan mengembalikan bounding box wajah terbesar."""
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) > 0:
        largest_face = max(faces, key=lambda face: face[2] * face[3])  # [x, y, w, h]
        return largest_face
    else:
        return None


def cropFace(image, face, target_size=(224, 224)):
    """Mencrop wajah dari gambar dengan menambahkan buffer dan resize dengan menjaga aspect ratio."""
    if face is not None:
        x, y, w, h = face
        buffer_top = int(h * 0.2)
        buffer_bottom = int(h * 0.2)
        buffer_left = int(w * 0.1)
        buffer_right = int(w * 0.1)
        y_top = max(0, y - buffer_top)
        y_bottom = min(image.shape[0], y + h + buffer_bottom)
        x_left = max(0, x - buffer_left)
        x_right = min(image.shape[1], x + w + buffer_right)
        cropped_face = image[y_top:y_bottom, x_left:x_right]
        old_h, old_w = cropped_face.shape[:2]
        target_w, target_h = target_size
        scale = min(target_w / old_w, target_h / old_h)
        new_w = int(old_w * scale)
        new_h = int(old_h * scale)
        resized_face = cv2.resize(cropped_face, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
        x_offset = (target_w - new_w) // 2
        y_offset = (target_h - new_h) // 2
        canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_face
        return canvas
    else:
        return None


def preprocessing(image):
    face = detectFace(image)
    if face is not None:
        cropped_face = cropFace(image, face)
        return cropped_face.astype(np.uint8)
    else:
        return None


def standarized_input(image):

    image = preprocessing(image)
    if image is not None:
        
        return image.astype(np.uint8)
    else:
        return None


In [ ]:
from skimage.feature import hog

def extract_hog_features(image, pixels_per_cell=(8, 8), cells_per_block=(2, 2), orientations=9):
    """Ekstraksi fitur HOG dari gambar."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Konversi ke grayscale
    features = hog(
        gray, 
        orientations=orientations, 
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block, 
        block_norm='L2-Hys', 
        visualize=False,
        transform_sqrt=True
    )
    return features

def preprocess_with_hog(img_list):
    """Preprocessing data dan ekstraksi HOG."""
    hog_features_list = []
    label_list = []

    for item in img_list:
        image = item[0]
        label = item[1]

        # Preprocessing dan standarisasi gambar
        std_img = standarized_input(image)
        if std_img is not None:
            # Ekstraksi fitur HOG
            features = extract_hog_features(std_img)
            hog_features_list.append(features)
            label_list.append(label)

    return hog_features_list, label_list


# Memanggil preprocess_with_hog pada dataset training dan testing
train_hog_features, train_labels = preprocess_with_hog(train_img)
test_hog_features, test_labels = preprocess_with_hog(test_img)


In [ ]:
import pandas as pd

def save_features_to_csv(features, labels, output_file):
    # Konversi fitur menjadi DataFrame
    features_df = pd.DataFrame(features)
    features_df['label'] = labels  # Tambahkan kolom label

    # Simpan ke CSV
    features_df.to_csv(output_file, index=False)
    print(f"Fitur dan label berhasil disimpan ke {output_file}")

# Simpan fitur training dan testing ke file CSV
save_features_to_csv(train_hog_features, train_labels, "hog_train_features.csv")
save_features_to_csv(test_hog_features, test_labels, "hog_test_features.csv")
